# RAG-over-PDF study tool — v1

This notebook is the "main" that wires the four pipeline stages together:

`parse → chunk → index → query`

Each stage lives in the `rag` package and writes a file to disk, so you can
inspect every intermediate artifact. v1 has **no OCR** and **no LLM** — the
output is a `context.txt` you paste into any chat model yourself.

**First time:** `pip install -r requirements.txt`. The embedding model
(~130 MB) downloads on first use and is cached afterwards.


## Setup
Put your slide PDFs in the `pdfs/` folder next to this notebook.

In [1]:
from pathlib import Path
import rag

# --- paths -------------------------------------------------------------
PDF_DIR   = Path("G:\\Ca' Foscari\\Semester 2\\")            # drop your .pdf slide decks here
WORK_DIR  = Path("artifacts")       # where intermediate files are written
WORK_DIR.mkdir(exist_ok=True)

RECORDS   = WORK_DIR / "records.jsonl"
CHUNKS    = WORK_DIR / "chunks.jsonl"
INDEX     = WORK_DIR / "index"          # -> index.npy + index.chunks.json
CONTEXT   = WORK_DIR / "context.txt"

# --- retrieval knobs ---------------------------------------------------
TOP_K  = 5     # how many independent chunks seed the result
WINDOW = 1     # how many neighbor slides (x-1, x+1) each seed drags in


## Stage 1 — Parse
PDFs → one record per slide (text + merged notes), saved as JSONL.

In [2]:
import json

records = rag.parse_directory(PDF_DIR)
rag.save_records(records, RECORDS)

n_docs  = len({r["doc_id"] for r in records})
n_empty = sum(1 for r in records if not r["has_text"])
print(f"Parsed {len(records)} slides across {n_docs} document(s).")
print(f"{n_empty} slide(s) had no extractable text (OCR candidates for later).")
print("\nExample record:")
print(json.dumps(records[0], indent=2, ensure_ascii=False)[:500])


Parsed 6162 slides across 73 document(s).
42 slide(s) had no extractable text (OCR candidates for later).

Example record:
{
  "doc_id": "Algorithms for massive data/01 intro",
  "slide_num": 1,
  "text": "Nicola Prezza\nAlgorithms for massive data\n1 - introduction",
  "source_path": "G:\\Ca' Foscari\\Semester 2\\Algorithms for massive data\\01 intro.pdf",
  "has_text": true,
  "has_notes": false
}


## Stage 2 — Chunk
v1 rule: one slide = one chunk. Empty (image-only) slides are dropped for now.

In [3]:
chunks = rag.chunk_records(records, drop_empty=True)
rag.save_records(chunks, CHUNKS)   # same JSONL writer works for chunks
print(f"{len(chunks)} chunks ready to index.")
print("Example chunk id:", chunks[0]["chunk_id"])


6120 chunks ready to index.
Example chunk id: Algorithms for massive data/01 intro::1


## Stage 3 — Embed & index
Embed every chunk with a local model and persist the matrix + chunks.

In [4]:
model = rag.load_model()                 # downloads on first run, then cached
index = rag.build_index(chunks, model)
rag.save_index(index, INDEX)
print(f"Indexed {len(index)} chunks with model '{index.model_name}'.")
print("Embedding matrix shape:", index.embeddings.shape)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/192 [00:00<?, ?it/s]

Indexed 6120 chunks with model 'BAAI/bge-small-en-v1.5'.
Embedding matrix shape: (6120, 384)


## Stage 4 — Query
Type your question below. The query is embedded, the top-k chunks retrieved,
their slide-neighbors pulled in, and the assembled context written to
`context.txt` (and printed). That file is your prompt — paste it into any chat
model along with your question.

In [5]:
question = "Explain how the chunking step relates to neighbor retrieval."

query_vec = rag.embed_query(question, model)
hits      = rag.retrieve(query_vec, index, k=TOP_K)

print("Top-k seeds:")
for chunk, score in hits:
    print(f"  {score:.3f}  {chunk['doc_id']} slide {chunk['slide_num']}")

expanded = rag.expand_neighbors(hits, index, window=WINDOW)
context  = rag.assemble_context(question, expanded)

CONTEXT.write_text(context, encoding="utf-8")
print(f"\nWrote {CONTEXT}  ({len(expanded)} chunks after neighbor expansion)\n")
print(context)


Top-k seeds:
  0.705  lecture4-indexconstruction slide 49
  0.692  lecture6-tfidf slide 3
  0.692  lecture6-tfidf (2) slide 3
  0.692  lecture6-tfidf (1) slide 3
  0.690  lecture4-indexconstruction slide 48

Wrote artifacts\context.txt  (13 chunks after neighbor expansion)

QUESTION:
Explain how the chunking step relates to neighbor retrieval.

CONTEXT:

[lecture4-indexconstruction - slide 47 | neighbor | score 0.690]
Other sorts of indexes
▪Positional indexes
▪Sort:  <termID, DocID, position>
▪Still sorting … just larger
▪Character n-gram indexes:
▪As text is parsed, enumerate n-grams
▪For each n-gram, need pointers to all dictionary terms 
containing it – the “postings”
▪… or postpone its construction once produced the 
complete lexicon
Sec. 4.5
for wildcards, spell checking, etc.
[NOTE] for wildcards, spell checking, etc.

[lecture4-indexconstruction - slide 48 | seed | score 0.690]
Sharding is a technique used in database systems and search engines 
to distribute/partition data (in

### One-call shortcut
Once the index exists you can skip the manual steps. `answer_to_file` runs
embed → retrieve → expand → write in one go.

In [22]:
rag.answer_to_file(
    "Given the lecture slides, derive a unified explanation of how prefix search in B‑trees, k‑gram intersection filtering, and multi‑head attention all rely on parallelizable operations, and discuss the computational trade‑offs across IR and deep learning.",
    index, model, CONTEXT, k=TOP_K, window=WINDOW,
)
print(CONTEXT.read_text(encoding="utf-8"))


QUESTION:
Given the lecture slides, derive a unified explanation of how prefix search in B‑trees, k‑gram intersection filtering, and multi‑head attention all rely on parallelizable operations, and discuss the computational trade‑offs across IR and deep learning.

CONTEXT:

[Information rtrival and web search/lecture2-vocabulary-postings - slide 26 | neighbor | score 0.778]
DOCUMENT INGESTION IN NEURAL IR

[Information rtrival and web search/lecture2-vocabulary-postings - slide 27 | seed | score 0.778]
Tokenization for NLP and Neural IR
▪Neural-IR: DeepNN to retrieve or re-rank search 
results in response to a query
▪Neural-IR, modern NLP and Generative AI are all based on 
special DeepNN architectures called Transformers
▪Texts are tokenized and then associated to low dimensional 
dense vectors (embeddings) before traversing the DeepNN
▪General architecture of retrieve/re-rank architecture
27
Inverted 
Index
Initial 
Retrieval
Texts
Ranked List
Candidate 
Texts
Queries
Reranker
Try to 

### Reusing a saved index
In a fresh session you don't need to re-parse. Load the index and the model,
then query:

```python
import rag
model = rag.load_model()
index = rag.load_index("artifacts/index")
rag.answer_to_file("your question", index, model, "artifacts/context.txt")
```


In [2]:
import rag
model = rag.load_model()
index = rag.load_index("artifacts/index")
#rag.answer_to_file("your question", index, model, "artifacts/context.txt")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [3]:
q = "Explain how k‑gram indexes, permuterm indexes, and multi‑head attention differ in the way they handle combinatorial explosion, and why each method uses normalization or scaling."
rag.answer_to_file(q, index, model, "artifacts/context.txt")

WindowsPath('artifacts/context.txt')